# author collection

In [ ]:
import pandas as pd

from tqdm.auto import tqdm
from sqlalchemy import select
from typing import Dict, Any, Optional

from scholar import S2Api, get_session, parse_date, pd_clean, get_corpus_id

from postgres.model import (
    Paper,
    Author,
    PaperAuthor
)
import os
from dotenv import load_dotenv
load_dotenv()

## api

In [ ]:
api = S2Api()

## db session

In [ ]:
session = get_session()

## update paper

In [ ]:
def update_paper(paper: Dict[str, Any]):
    """
    最小化保存 Paper。

    主要用于 authorpaper 接口返回的 paper。
    这些 paper 的作用是构建 co-author graph，
    不直接参与 relevance score，所以不需要保存完整字段。

    保存字段：
    - corpus_id
    - title
    - publication_date
    """
    corpus_id = get_corpus_id(paper)

    if not corpus_id:
        return None

    obj = session.get(Paper, corpus_id)

    if obj is None:
        obj = Paper(
            corpus_id=corpus_id,
            title=pd_clean(paper.get("title")) or "(untitled)",
        )
        session.add(obj)

    title = pd_clean(paper.get("title"))
    if title:
        obj.title = title

    publication_date = parse_date(paper.get("publicationDate"))
    if publication_date:
        obj.publication_date = publication_date

    session.flush()

    return corpus_id

## update author

In [ ]:
def get_author_id(author: Dict[str, Any]):
    """
    从 S2 author 对象中提取 author_id。

    数据库中 author_id / paper_author.author_id 是 BIGINT，
    所以这里统一转成 int。
    """
    if not author:
        return None

    author_id = author.get("authorId") or author.get("author_id")

    if not author_id:
        return None

    return int(author_id)

In [ ]:
def update_author(author: Dict[str, Any]):
    """
    新增或更新 Author 表。

    作用：
    - 保存作者基本信息
    - 后续通过 PaperAuthor 构建 co-author graph
    """
    if not author:
        return None

    author_id = get_author_id(author)

    if not author_id:
        return None

    author_id = int(author_id)

    obj = session.get(Author, author_id)

    if obj is None:
        obj = Author(author_id=author_id)
        session.add(obj)

    obj.name = pd_clean(author.get("name"))

    if hasattr(obj, "affiliations"):
        obj.affiliations = author.get("affiliations") or []

    if hasattr(obj, "homepage"):
        obj.homepage = author.get("homepage") or ""

    if hasattr(obj, "url"):
        obj.url = author.get("url") or ""

    if hasattr(obj, "h_index"):
        obj.h_index = (
            author.get("hIndex")
            or author.get("h_index")
            or 0
        )

    if hasattr(obj, "paper_count"):
        obj.paper_count = (
            author.get("paperCount")
            or author.get("paper_count")
            or 0
        )

    if hasattr(obj, "citation_count"):
        obj.citation_count = (
            author.get("citationCount")
            or author.get("citation_count")
            or 0
        )

    if hasattr(obj, "external_ids"):
        obj.external_ids = (
            author.get("externalIds")
            or author.get("external_ids")
            or {}
        )

    session.flush()

    return author_id

## update paper author

In [ ]:
def update_paper_author(
    paper_id: int,
    author_id: int,
    author_order: int,
):
    """
    新增 PaperAuthor 关系。

    注意：
    paper_id 是 BIGINT
    author_id 也是 BIGINT
    不能把 author_id 转成 str。
    """
    if not paper_id or not author_id:
        return None

    paper_id = int(paper_id)
    author_id = int(author_id)

    exists = session.execute(
        select(PaperAuthor)
        .where(PaperAuthor.paper_id == paper_id)
        .where(PaperAuthor.author_id == author_id)
    ).scalar_one_or_none()

    if exists:
        return exists

    obj = PaperAuthor(
        paper_id=paper_id,
        author_id=author_id,
        author_order=author_order,
    )

    session.add(obj)
    session.flush()

    return obj

In [ ]:
def update_paper_authors(paper: Dict[str, Any]):
    """
    保存一篇 paper 及其 authors。

    写入：
    - Paper
    - Author
    - PaperAuthor

    返回：
    {
        "paper_id": corpus_id,
        "author_ids": [...]
    }
    """
    paper_id = update_paper(paper)

    if not paper_id:
        return {
            "paper_id": None,
            "author_ids": [],
        }

    author_ids = []

    for author_order, author in enumerate(paper.get("authors") or [], start=1):
        author_id = update_author(author)

        if not author_id:
            continue

        update_paper_author(
            paper_id=paper_id,
            author_id=author_id,
            author_order=author_order,
        )

        author_ids.append(author_id)

    return {
        "paper_id": paper_id,
        "author_ids": author_ids,
    }

## collect author all papers

In [ ]:
expanded_author_ids = set()


def collect_author_all_papers(author: Dict[str, Any]):
    """
    收集一个 author 的所有 papers。

    数据来源：
    - api.author_papers(author_id)

    写入：
    - Paper
    - Author
    - PaperAuthor

    提交策略：
    - 每完成一个 author 的 author_papers 保存，就 commit 一次
    - 这样单篇 seed 耗时很长时，中间结果也不会丢失

    注意：
    - 只扩展当前 author
    - 保存 author_papers 返回论文中的所有 authors
    - 不继续递归扩展这些 coauthors
    """
    author_id = update_author(author)

    if not author_id:
        return {
            "author_id": None,
            "paper_count": 0,
            "author_count": 0,
            "status": "missing_author_id",
        }

    author_id = int(author_id)

    if author_id in expanded_author_ids:
        return {
            "author_id": author_id,
            "paper_count": 0,
            "author_count": 0,
            "status": "skipped",
        }

    try:
        papers = api.author_papers(author_id) or []

        paper_count = 0
        author_count = 0

        for paper in papers:
            result = update_paper_authors(paper)

            if not result["paper_id"]:
                continue

            paper_count += 1
            author_count += len(result["author_ids"])

        # 关键：每个 author 完成后立即提交
        session.commit()

        expanded_author_ids.add(author_id)

        return {
            "author_id": author_id,
            "paper_count": paper_count,
            "author_count": author_count,
            "status": "ok",
        }

    except Exception as e:
        # 当前 author 失败，只回滚当前 author 这批写入
        session.rollback()

        return {
            "author_id": author_id,
            "paper_count": 0,
            "author_count": 0,
            "status": "error",
            "error": str(e),
        }

## short text for paper title

In [ ]:
def short_text(text, max_len: int = 80):
    """
    缩短标题，避免 tqdm 显示太长。
    """
    text = pd_clean(text) or ""

    if len(text) <= max_len:
        return text

    return text[:max_len] + "..."

In [ ]:
def has_valid_context(citation: Dict[str, Any]):
    """
    判断一条 citation 是否包含有效 context。

    后续 relevance / author independence 都只基于有 context 的 citation，
    所以这里没有 context 的 citingPaper 直接跳过，不入库。
    """
    if not citation:
        return False

    contexts = citation.get("contexts") or citation.get("context") or []

    if isinstance(contexts, str):
        return bool(pd_clean(contexts))

    if isinstance(contexts, list):
        return any(pd_clean(x) for x in contexts)

    return False

In [ ]:
def collect_authors_all_papers_from_paper(paper: Dict[str, Any]):
    """
    对一篇 paper 中的每个 author 调用 author_papers。

    用途：
    - seed paper authors
    - citing paper authors

    返回统计信息。
    """
    authors = paper.get("authors") or []

    authorpaper_call_count = 0
    authorpaper_paper_count = 0
    authorpaper_author_count = 0

    for author in authors:
        result = collect_author_all_papers(author)

        if result["status"] == "ok":
            authorpaper_call_count += 1
            authorpaper_paper_count += result["paper_count"]
            authorpaper_author_count += result["author_count"]

    return {
        "authorpaper_call_count": authorpaper_call_count,
        "authorpaper_paper_count": authorpaper_paper_count,
        "authorpaper_author_count": authorpaper_author_count,
    }

In [ ]:
def get_seed_identifier(row: Dict[str, Any]):
    """
    从 seed_papers_final.xlsx 的一行中提取 S2 API identifier。

    优先级：
    1. corpus_id / corpusId / id
    2. paper_id / paperId
    3. doi / DOI
    """
    for col in ["corpus_id", "corpusId", "id"]:
        if col in row and pd.notna(row[col]):
            return f"CorpusId:{int(row[col])}"

    for col in ["paper_id", "paperId"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col]).strip()

    for col in ["doi", "DOI"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col]).strip()

    return None

In [ ]:
def collect_unique_authors_all_papers(
    authors_by_id: Dict[int, Dict[str, Any]],
    seed_index: int,
    seed_title: str = "",
):
    """
    对当前 seed 相关的 unique authors 调用 api.author_papers。

    子 tqdm 显示：
    - 当前 seed index
    - 当前 seed title
    - 当前 author_id / author name

    提交策略：
    - collect_author_all_papers() 内部每个 author commit 一次
    """
    authorpaper_call_count = 0
    authorpaper_paper_count = 0
    authorpaper_author_count = 0
    authorpaper_skipped_count = 0
    authorpaper_error_count = 0

    items = list(authors_by_id.items())

    desc = f"seed {seed_index} authors"
    if seed_title:
        desc = f"seed {seed_index} authors | {short_text(seed_title, 35)}"

    pbar = tqdm(
        items,
        desc=desc,
        leave=False,
        dynamic_ncols=True,
    )

    for author_id, author in pbar:
        author_name = pd_clean(author.get("name")) or ""

        pbar.set_postfix(
            {
                "author_id": author_id,
                "name": author_name[:20],
            }
        )

        result = collect_author_all_papers(author)

        if result["status"] == "ok":
            authorpaper_call_count += 1
            authorpaper_paper_count += result["paper_count"]
            authorpaper_author_count += result["author_count"]
        elif result["status"] == "skipped":
            authorpaper_skipped_count += 1
        elif result["status"] == "error":
            authorpaper_error_count += 1
            tqdm.write(
                f"[seed {seed_index}] author error | "
                f"author_id={str(result.get('author_id'))} | "
                f"error={result.get('error')}"
            )

    return {
        "authorpaper_call_count": authorpaper_call_count,
        "authorpaper_paper_count": authorpaper_paper_count,
        "authorpaper_author_count": authorpaper_author_count,
        "authorpaper_skipped_count": authorpaper_skipped_count,
        "authorpaper_error_count": authorpaper_error_count,
    }

In [ ]:
def get_citing_paper(citation: Dict[str, Any]):
    """
    从 citation item 中提取 citingPaper。

    兼容两种格式：
    1. 原始 S2 格式：citation["citingPaper"]
    2. 如果 S2Api 已经展开，则 citation 本身就是 citing paper
    """
    if not citation:
        return {}

    return citation.get("citingPaper") or citation

In [ ]:
def collect_one_seed_author_relations(
    row: Dict[str, Any],
    seed_index: int
):
    """
    处理一篇 seed paper 的作者关系。

    写入：
    - seed paper
    - seed paper authors
    - seed paper-author
    - 有 context 的 citing papers
    - 有 context 的 citing paper authors
    - 有 context 的 citing paper-author
    - seed/citing authors 的所有 papers

    不写入：
    - Citation
    - CSV
    - progress table
    """
    seed_identifier = get_seed_identifier(row)

    if not seed_identifier:
        print(f"[{seed_index}] missing seed identifier")
        return {
            "seed_index": seed_index,
            "seed_identifier": None,
            "seed_title": None,
            "seed_corpus_id": None,
            "seed_author_count": 0,
            "citation_item_count": 0,
            "context_citation_count": 0,
            "citing_paper_count": 0,
            "citing_author_count": 0,
            "unique_author_count": 0,
            "authorpaper_call_count": 0,
            "authorpaper_paper_count": 0,
            "authorpaper_author_count": 0,
            "authorpaper_skipped_count": 0,
            "status": "missing_seed_identifier",
        }

    seed_paper = api.paper_detail(seed_identifier)
    seed_title = short_text(seed_paper.get("title"), 40)

    if not seed_paper:
        print(f"[{seed_index}] missing seed detail: {seed_identifier}")
        return {
            "seed_index": seed_index,
            "seed_identifier": seed_identifier,
            "seed_title": None,
            "seed_corpus_id": None,
            "seed_author_count": 0,
            "citation_item_count": 0,
            "context_citation_count": 0,
            "citing_paper_count": 0,
            "citing_author_count": 0,
            "unique_author_count": 0,
            "authorpaper_call_count": 0,
            "authorpaper_paper_count": 0,
            "authorpaper_author_count": 0,
            "authorpaper_skipped_count": 0,
            "status": "missing_seed_detail",
        }

    # 保存 seed paper + seed authors + seed paper-author
    seed_result = update_paper_authors(seed_paper)
    seed_corpus_id = seed_result["paper_id"]

    if not seed_corpus_id:
        print(f"[{seed_index}] missing seed corpus id: {seed_identifier}")
        return {
            "seed_index": seed_index,
            "seed_identifier": seed_identifier,
            "seed_title": None,
            "seed_corpus_id": None,
            "seed_author_count": 0,
            "citation_item_count": 0,
            "context_citation_count": 0,
            "citing_paper_count": 0,
            "citing_author_count": 0,
            "unique_author_count": 0,
            "authorpaper_call_count": 0,
            "authorpaper_paper_count": 0,
            "authorpaper_author_count": 0,
            "authorpaper_skipped_count": 0,
            "status": "missing_seed_corpus_id",
        }

    # 当前 seed 下所有需要扩展 author_papers 的 unique authors
    authors_by_id = {}

    for author in seed_paper.get("authors") or []:
        author_id = get_author_id(author)
        if author_id:
            authors_by_id[int(author_id)] = author

    # 读取 citations
    citations = api.paper_citations(seed_identifier) or []

    citation_item_count = len(citations)
    context_citation_count = 0
    citing_paper_count = 0
    citing_author_count = 0

    # 这里只处理有 context 的 citation
    for citation in tqdm(
        citations,
        desc=f"seed {seed_index} ({seed_title}) citing papers",
        leave=False,
        dynamic_ncols=True,
    ):
        if not has_valid_context(citation):
            continue

        context_citation_count += 1

        citing_paper = get_citing_paper(citation)

        if not citing_paper:
            continue

        # 保存有 context 的 citing paper + authors + paper-author
        citing_result = update_paper_authors(citing_paper)

        if not citing_result["paper_id"]:
            continue

        citing_paper_count += 1
        citing_author_count += len(citing_result["author_ids"])

        for author in citing_paper.get("authors") or []:
            author_id = get_author_id(author)
            if author_id:
                authors_by_id[int(author_id)] = author

    # 子 tqdm：对当前 seed 的 unique authors 统一扩展 author_papers
    authorpaper_result = collect_unique_authors_all_papers(
        authors_by_id=authors_by_id,
        seed_index=seed_index,
        seed_title=seed_title
    )

    print(
        f"[{seed_index}] ok | "
        f"seed={seed_corpus_id} | "
        f"seed_title={seed_title} | "
        f"seed_authors={len(seed_result['author_ids'])} | "
        f"citations={citation_item_count} | "
        f"context_citations={context_citation_count} | "
        f"citing_papers={citing_paper_count} | "
        f"unique_authors={len(authors_by_id)} | "
        f"authorpaper_calls={authorpaper_result['authorpaper_call_count']} | "
        f"authorpaper_papers={authorpaper_result['authorpaper_paper_count']} | "
        f"skipped_authors={authorpaper_result['authorpaper_skipped_count']} | "
        f"authorpaper_errors={authorpaper_result['authorpaper_error_count']}"
    )

    return {
        "seed_index": seed_index,
        "seed_identifier": seed_identifier,
        "seed_title": seed_title,
        "seed_corpus_id": seed_corpus_id,
        "seed_author_count": len(seed_result["author_ids"]),
        "citation_item_count": citation_item_count,
        "context_citation_count": context_citation_count,
        "citing_paper_count": citing_paper_count,
        "citing_author_count": citing_author_count,
        "unique_author_count": len(authors_by_id),
        "authorpaper_call_count": authorpaper_result["authorpaper_call_count"],
        "authorpaper_paper_count": authorpaper_result["authorpaper_paper_count"],
        "authorpaper_author_count": authorpaper_result["authorpaper_author_count"],
        "authorpaper_skipped_count": authorpaper_result["authorpaper_skipped_count"],
        "authorpaper_error_count": authorpaper_result["authorpaper_error_count"],
        "status": "ok",
    }

## run

In [ ]:
def run_author_collection(
    seed_file: str = "sample/seed_papers_final.xlsx",
    start_index: int = 1,
    end_index: Optional[int] = None
):
    """
    author relation collection 最终入口。

    特点：
    - 只写 Paper / Author / PaperAuthor
    - 不写 Citation
    - 不写 CSV
    - 不建 progress table
    - 每篇 seed 单独 commit
    - 用 tqdm 和 print 显示进度
    - 对 seed/citing paper 中出现的每个 author 调用 api.author_papers(author_id)
    - 可用 start_index 手动续跑
    """
    df_seed = pd.read_excel(seed_file)
    records = df_seed.to_dict("records")

    if end_index is None:
        end_index = len(records)

    selected_records = records[start_index - 1:end_index]

    logs = []

    for offset, row in enumerate(
        tqdm(selected_records, desc="collect author relations"),
        start=0,
    ):
        seed_index = start_index + offset

        try:
            log = collect_one_seed_author_relations(
                row=row,
                seed_index=seed_index
            )

            session.commit()

        except Exception as e:
            session.rollback()

            log = {
                "seed_index": seed_index,
                "seed_identifier": get_seed_identifier(row),
                "seed_corpus_id": None,
                "seed_author_count": 0,
                "citation_item_count": 0,
                "context_citation_count": 0,
                "citing_paper_count": 0,
                "citing_author_count": 0,
                "unique_author_count": 0,
                "authorpaper_call_count": 0,
                "authorpaper_paper_count": 0,
                "authorpaper_author_count": 0,
                "authorpaper_skipped_count": 0,
                "status": "error",
                "error": str(e),
            }

            print(f"[{seed_index}] error: {e}")

        logs.append(log)

    result = {
        "log": pd.DataFrame(logs),
        "seed_file": seed_file,
        "start_index": start_index,
        "end_index": end_index,
        "ok_count": sum(1 for x in logs if x.get("status") == "ok"),
        "error_count": sum(1 for x in logs if x.get("status") == "error"),
        "expanded_author_count": len(expanded_author_ids),
    }

    return result

In [ ]:
run_author_collection(seed_file="sample/seed_papers_final.xlsx")